<a href="https://colab.research.google.com/github/dksree/elevance-skills-internship/blob/main/task6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import plotly.express as px
from datetime import datetime
import pytz

In [ ]:
df = pd.read_csv("google_play_store_dataset.csv")

df.columns = df.columns.str.strip()

df.head()

,App,Category,Rating,Reviews,Size,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Country
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up,India
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,India
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up,India
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up,India
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up,India


In [ ]:
df['Reviews'] = pd.to_numeric(df['Reviews'], errors='coerce')
df['Rating'] = pd.to_numeric(df['Rating'], errors='coerce')

df['Size'] = df['Size'].astype(str).str.replace('M','')
df['Size'] = pd.to_numeric(df['Size'], errors='coerce')

df['Last Updated'] = pd.to_datetime(df['Last Updated'], errors='coerce')

In [ ]:
df_filtered = df[
    (df['Rating'] >= 4.2) &
    (~df['App'].str.contains(r'\d', na=False)) &
    (df['Category'].str.startswith(('T','P'), na=False)) &
    (df['Reviews'] > 1000) &
    (df['Size'].between(20,80))
]

In [ ]:
translations = {
    "TRAVEL & LOCAL": "Voyage et Local",
    "PRODUCTIVITY": "Productividad",
    "PHOTOGRAPHY": "写真"
}

df_filtered['Category'] = df_filtered['Category'].str.upper().replace(translations)

/tmp/ipykernel_195/1878321215.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['Category'] = df_filtered['Category'].str.upper().replace(translations)


In [ ]:
df_filtered['Month'] = df_filtered['Last Updated'].dt.to_period('M').dt.to_timestamp()

/tmp/ipykernel_195/679812416.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['Month'] = df_filtered['Last Updated'].dt.to_period('M').dt.to_timestamp()


In [ ]:
time_data = df_filtered.groupby(['Month','Category'])['Reviews'].sum().reset_index()

time_data.head()

,Month,Category,Reviews
0,2014-11-01,写真,50893.0
1,2016-10-01,Productividad,40328.0
2,2016-12-01,PERSONALIZATION,117231.0
3,2017-03-01,写真,1494491.0
4,2017-06-01,写真,197295.0


In [ ]:
time_data['Growth'] = time_data.groupby('Category')['Reviews'].pct_change()

In [ ]:
fig = px.area(
    time_data,
    x="Month",
    y="Reviews",
    color="Category",
    title="Cumulative Installs Over Time by App Category"
)

In [ ]:
growth_data = time_data[time_data['Growth'] > 0.25]

fig.add_scatter(
    x=growth_data['Month'],
    y=growth_data['Reviews'],
    mode='markers',
    marker=dict(size=10),
    name="High Growth"
)

In [ ]:
ist = pytz.timezone('Asia/Kolkata')
current_time = datetime.now(ist)

if 16 <= current_time.hour < 18:
    fig.show()
else:
    print("Graph only visible between 4 PM and 6 PM IST")

Graph only visible between 4 PM and 6 PM IST
